<a href="https://colab.research.google.com/github/M7office/Stroke/blob/main/AHA_module_time_dependence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Slide 4 / Figure 4 — Time dependence of biological-module values
# Colab-ready script
#
# Input files expected in the current Colab folder (/content):
#   C_patient_data*.csv
#   C_NPX_data*.csv
#   strokecog_literature_aligned_pathway_framework*.csv
#   strokecog_literature_aligned_protein_pathway_assignment_summary*.csv
#
# Output folder:
#   AHA_slide04_outputs/
# ============================================================

from pathlib import Path
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -----------------------------
# User settings
# -----------------------------
BASE = Path.cwd()  # Colab default: /content
OUT = BASE / "AHA_slide04_outputs_v8"
OUT.mkdir(parents=True, exist_ok=True)

SIS3_CUTOFF = 63
RANDOM_SEED = 42
N_BOOT = 5000

PATHWAY_ASSIGNMENT_MODE = "all"  # "all" or "primary"
FILTER_LOW_DETECTION_PROTEINS = True
LOW_DETECTION_FILTER_MODE = "complete_case"

TIME_ORDER = ["0–1y", "1–3y", ">3y"]
TIME_BIN_EDGES = [0, 1, 3, np.inf]
TIME_BIN_LABELS = TIME_ORDER

PATHWAY_ORDER_FIXED = [
    "Serotonin / tryptophan-kynurenine / monoamine metabolism",
    "Vesicle secretion / extracellular vesicle / membrane trafficking",
    "Cell death / cellular stress / proteostasis",
    "Metabolic / lipid / atherosclerosis / mitochondrial-energy biology",
    "mTOR / MAPK / NF-kB / growth-survival signaling",
    "Hormone / neuroendocrine / HPA-like systemic signaling",
    "Systemic organ injury / leakage / comorbidity markers",
    "Peripheral immune / inflammatory activation",
    "Complement / coagulation / platelet axis",
    "Endothelial / BBB / neurovascular unit",
    "Synaptic / neuronal plasticity / neurotrophic signaling",
    "Integrin / ECM / cell adhesion / vascular remodeling",
]

PATHWAY_SHORT = {
    "Serotonin / tryptophan-kynurenine / monoamine metabolism": "Serotonin / kynurenine / monoamine",
    "Vesicle secretion / extracellular vesicle / membrane trafficking": "Vesicle / extracellular vesicle trafficking",
    "Cell death / cellular stress / proteostasis": "Cell death / stress / proteostasis",
    "Metabolic / lipid / atherosclerosis / mitochondrial-energy biology": "Metabolic / lipid / mitochondrial-energy",
    "mTOR / MAPK / NF-kB / growth-survival signaling": "mTOR / MAPK / NF-κB signaling",
    "Hormone / neuroendocrine / HPA-like systemic signaling": "Hormone / neuroendocrine signaling",
    "Systemic organ injury / leakage / comorbidity markers": "Systemic injury / comorbidity markers",
    "Peripheral immune / inflammatory activation": "Peripheral immune / inflammatory activation",
    "Complement / coagulation / platelet axis": "Complement / coagulation / platelet axis",
    "Endothelial / BBB / neurovascular unit": "Endothelial / BBB / neurovascular unit",
    "Synaptic / neuronal plasticity / neurotrophic signaling": "Synaptic / neuroplasticity / neurotrophic",
    "Integrin / ECM / cell adhesion / vascular remodeling": "Integrin / ECM / vascular remodeling",
}

COLORS = {
    "lower": "#D55E00",
    "higher": "#0072B2",
    "effect": "#2E5C68",
    "black": "#303030",
    "gray": "#6E6E6E",
    "verylight": "#EAEAEA",
    "lightgray": "#D9D9D9",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.8,
    "ytick.labelsize": 8.8,
    "legend.fontsize": 8.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 450,
})

# -----------------------------
# Helper functions
# -----------------------------
def read_csv_safely(path: Path) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            df.columns = df.columns.astype(str).str.strip()
            return df
        except UnicodeDecodeError:
            pass
    df = pd.read_csv(path)
    df.columns = df.columns.astype(str).str.strip()
    return df


def csv_files():
    return sorted(BASE.glob("*.csv"))


def find_csv(contains_all=None, contains_any=None, required=True, label="file", exclude_any=None):
    contains_all = [x.lower() for x in (contains_all or [])]
    contains_any = [x.lower() for x in (contains_any or [])]
    exclude_any = [x.lower() for x in (exclude_any or [])]
    matches = []
    for f in csv_files():
        name = f.name.lower()
        if contains_all and not all(x in name for x in contains_all):
            continue
        if contains_any and not any(x in name for x in contains_any):
            continue
        if exclude_any and any(x in name for x in exclude_any):
            continue
        matches.append(f)
    if not matches:
        if required:
            raise FileNotFoundError(f"Could not find {label}. Tried all={contains_all}, any={contains_any}")
        return None
    return sorted(matches, key=lambda p: (len(p.name), p.name))[0]


def find_col(df, candidates, required=True, label="column", avoid=None):
    avoid = [a.lower() for a in (avoid or [])]
    lower_to_col = {c.lower(): c for c in df.columns}
    for cand in candidates:
        c = lower_to_col.get(cand.lower())
        if c is not None and not any(a in c.lower() for a in avoid):
            return c
    for c in df.columns:
        c_l = c.lower()
        if any(cand.lower() in c_l for cand in candidates):
            if not any(a in c_l for a in avoid):
                return c
    if required:
        raise ValueError(f"Could not find {label}. Tried {candidates}. Available columns: {df.columns.tolist()}")
    return None


def normalize_id_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.0$", "", s)
    return s


def normalize_oid_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    m = re.search(r"(OID\d+)", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    return s


def normalize_feature_column_name(c):
    s = str(c).strip()
    m = re.search(r"(OID\d+)", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    return s


def unique_preserve(seq):
    out, seen = [], set()
    for x in seq:
        if pd.isna(x):
            continue
        x = str(x).strip()
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out


def detect_time_years(series, colname=""):
    x = pd.to_numeric(series, errors="coerce")
    lname = str(colname).lower()
    max_val = np.nanmax(x.values) if np.isfinite(x).any() else np.nan
    if "year" in lname or "yrs" in lname or "yr" in lname:
        return x
    if "day" in lname or (np.isfinite(max_val) and max_val > 40):
        return x / 365.25
    if "month" in lname or (np.isfinite(max_val) and max_val > 6):
        return x / 12.0
    return x


def time_bin_from_years(y):
    if pd.isna(y):
        return np.nan
    if 0 <= y < 1:
        return "0–1y"
    if 1 <= y < 3:
        return "1–3y"
    if y >= 3:
        return ">3y"
    return np.nan


def zscore_df(df):
    numeric = df.apply(pd.to_numeric, errors="coerce")
    return (numeric - numeric.mean(axis=0)) / numeric.std(axis=0, ddof=0).replace(0, np.nan)


def bootstrap_mean_diff_stats(a, b, n_boot=N_BOOT, seed=RANDOM_SEED):
    """
    Returns mean(a) - mean(b), 95% CI, bootstrap p-value.
    """
    a = pd.Series(a).dropna().astype(float).values
    b = pd.Series(b).dropna().astype(float).values
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan, np.nan

    diff = float(np.mean(a) - np.mean(b))
    rng = np.random.default_rng(seed)

    boot = np.empty(n_boot)
    for i in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boot[i] = np.mean(aa) - np.mean(bb)
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    pooled_mean = (np.sum(a) + np.sum(b)) / (len(a) + len(b))
    a_null = a - np.mean(a) + pooled_mean
    b_null = b - np.mean(b) + pooled_mean
    boot_null = np.empty(n_boot)
    for i in range(n_boot):
        aa = rng.choice(a_null, size=len(a_null), replace=True)
        bb = rng.choice(b_null, size=len(b_null), replace=True)
        boot_null[i] = np.mean(aa) - np.mean(bb)
    p_boot = float(np.mean(np.abs(boot_null) >= abs(diff)))
    return diff, float(ci_low), float(ci_high), p_boot


def fmt_p3(p):
    if pd.isna(p):
        return "NA"
    return f"{float(p):.3f}"


def wrap_caption(text, width=175):
    return "\n".join(textwrap.wrap(text, width=width, break_long_words=False))


def symmetric_axis_limit(values, min_limit=0.4, step=0.1, pad=0.04):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return min_limit
    lim = np.max(np.abs(vals)) + pad
    lim = max(min_limit, lim)
    lim = np.ceil(lim / step) * step
    return float(lim)


# -----------------------------
# 1. Load and merge data
# -----------------------------
print("Current folder:", BASE)
print("CSV files found:")
for f in csv_files():
    print(" -", f.name)

patient_file = find_csv(contains_any=["patient"], required=True, label="patient file")
npx_file = find_csv(contains_any=["npx"], required=True, label="NPX file")
framework_file = find_csv(contains_all=["pathway_framework"], required=True, label="pathway framework file")
assignment_file = find_csv(contains_all=["protein_pathway_assignment"], required=True, label="protein pathway assignment file")

patient = read_csv_safely(patient_file)
npx = read_csv_safely(npx_file)
framework = read_csv_safely(framework_file)
assign = read_csv_safely(assignment_file)

patient_id_col = find_col(
    patient,
    ["patient_id", "patientid", "participant_id", "subject_id", "sample_id", "record_id", "pid", "id"],
    required=False,
    label="patient ID column",
    avoid=["sis", "score", "time", "age"],
)
if patient_id_col is None:
    patient_id_col = patient.columns[0]

sis3_col = find_col(patient, ["sis3", "sis_3", "sis 3"], label="SIS3 column")
time_col = find_col(patient, ["time_year", "time_years", "timesince", "years", "month", "day", "time_since", "timesince", "time"], label="time column")

patient = patient.copy()
patient["_patient_id_norm"] = patient[patient_id_col].map(normalize_id_value)
patient["sis3"] = pd.to_numeric(patient[sis3_col], errors="coerce")
patient["time_years"] = detect_time_years(patient[time_col], time_col)
patient["time_bin"] = patient["time_years"].map(time_bin_from_years)

ref_ids = set(patient["_patient_id_norm"].dropna().astype(str))
id_col_npx = None
best_overlap = -1
for c in npx.columns:
    vals = set(npx[c].map(normalize_id_value).dropna().astype(str))
    overlap = len(vals.intersection(ref_ids))
    if overlap > best_overlap:
        best_overlap = overlap
        id_col_npx = c

npx_value_col = find_col(npx, ["npx", "value"], required=False, label="NPX value column")
oid_col_npx = find_col(npx, ["oid", "olinkid", "assay", "protein_id"], required=False, label="OID column in NPX file")

if best_overlap > 0 and npx_value_col is not None and oid_col_npx is not None and id_col_npx != oid_col_npx:
    npx["_patient_id_norm"] = npx[id_col_npx].map(normalize_id_value)
    npx["_OID_norm"] = npx[oid_col_npx].map(normalize_oid_value)
    npx_wide = npx.pivot_table(index="_patient_id_norm", columns="_OID_norm", values=npx_value_col, aggfunc="mean").reset_index()
    merged = patient.merge(npx_wide, on="_patient_id_norm", how="inner")
elif best_overlap > 0:
    npx_wide = npx.rename(columns={c: normalize_feature_column_name(c) for c in npx.columns})
    id_col_norm = normalize_feature_column_name(id_col_npx)
    npx_wide["_patient_id_norm"] = npx_wide[id_col_norm].map(normalize_id_value)
    merged = patient.merge(npx_wide, on="_patient_id_norm", how="inner")
else:
    if len(npx) != len(patient):
        raise ValueError("Could not match patient IDs, and NPX row count does not match patient row count.")
    npx_wide = npx.rename(columns={c: normalize_feature_column_name(c) for c in npx.columns})
    npx_wide["_row_order"] = np.arange(len(npx_wide))
    patient["_row_order"] = np.arange(len(patient))
    merged = patient.merge(npx_wide, on="_row_order", how="inner")
    merged["_patient_id_norm"] = merged["_patient_id_norm"].fillna(merged["_row_order"].astype(str))
    print("\nNote: using row-order alignment because NPX file had no matching patient ID column.")

if merged.empty:
    raise ValueError("Patient and NPX files did not merge.")

oid_col_assign = find_col(assign, ["OID", "oid", "olinkid", "protein_id"], label="OID column in assignment file")
if PATHWAY_ASSIGNMENT_MODE == "primary":
    pathway_col_assign = find_col(assign, ["primary_literature_aligned_pathway_draft", "primary_literature_aligned_pathway", "pathway"], label="primary pathway column")
    mapping = assign[[oid_col_assign, pathway_col_assign]].copy()
    mapping.columns = ["OID", "pathway"]
elif PATHWAY_ASSIGNMENT_MODE == "all":
    pathway_col_assign = find_col(assign, ["all_literature_aligned_pathways_draft", "all_literature_aligned_pathways", "all_pathways", "pathways"], label="all-pathways column")
    mapping = assign[[oid_col_assign, pathway_col_assign]].copy()
    mapping.columns = ["OID", "pathway"]
    mapping["pathway"] = mapping["pathway"].astype(str).str.split(r"\s*[;|,]\s*", regex=True)
    mapping = mapping.explode("pathway")
else:
    raise ValueError("PATHWAY_ASSIGNMENT_MODE must be 'primary' or 'all'.")

mapping["OID"] = mapping["OID"].map(normalize_oid_value)
mapping["pathway"] = mapping["pathway"].astype(str).str.strip()
mapping = mapping.dropna()
mapping = mapping[mapping["pathway"].isin(PATHWAY_ORDER_FIXED)]

protein_cols = [c for c in merged.columns if re.fullmatch(r"OID\d+", str(c), flags=re.IGNORECASE)]
protein_cols = [normalize_feature_column_name(c) for c in protein_cols]
protein_cols = [c for c in protein_cols if c in merged.columns]
protein_numeric = merged[protein_cols].apply(pd.to_numeric, errors="coerce")

if FILTER_LOW_DETECTION_PROTEINS and LOW_DETECTION_FILTER_MODE == "complete_case":
    detection_rate = protein_numeric.notna().sum(axis=0) / len(protein_numeric)
    retained = detection_rate[detection_rate == 1.0].index.tolist()
    print(f"\nProteins before complete-case filter: {len(protein_cols)}")
    print(f"Proteins retained after complete-case filter: {len(retained)}")
    protein_cols = retained
    protein_numeric = protein_numeric[protein_cols]
    mapping = mapping[mapping["OID"].isin(protein_cols)].copy()
else:
    mapping = mapping[mapping["OID"].isin(protein_cols)].copy()

if mapping.empty:
    raise ValueError("No pathway-assignment OIDs matched retained NPX protein columns.")

protein_z = zscore_df(protein_numeric)
score_df = merged[["_patient_id_norm", "sis3", "time_years", "time_bin"]].copy()
for pathway in PATHWAY_ORDER_FIXED:
    oids = sorted(set(mapping.loc[mapping["pathway"] == pathway, "OID"]).intersection(protein_z.columns))
    score_df[pathway] = protein_z[oids].mean(axis=1) if len(oids) else np.nan

score_df = score_df.dropna(subset=["sis3", "time_bin"]).copy()
score_df["time_bin"] = pd.Categorical(score_df["time_bin"], categories=TIME_ORDER, ordered=True)
score_df["sis3_group"] = np.where(score_df["sis3"] <= SIS3_CUTOFF, "Lower SIS3 (≤63)", "Higher SIS3 (>63)")
score_df.to_csv(OUT / "slide04_patient_level_module_values.csv", index=False)

# -----------------------------
# 2. Summaries for bar plots
# -----------------------------
bar_rows = []
for sis3_group in ["Lower SIS3 (≤63)", "Higher SIS3 (>63)"]:
    for pathway in PATHWAY_ORDER_FIXED:
        row = {
            "SIS3 group": sis3_group,
            "Biological module": pathway,
            "Biological module, short": PATHWAY_SHORT.get(pathway, pathway),
        }
        for t in TIME_ORDER:
            vals = pd.to_numeric(score_df.loc[(score_df["sis3_group"] == sis3_group) & (score_df["time_bin"] == t), pathway], errors="coerce").dropna()
            row[f"{t}, mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{t}, n"] = int(len(vals))
        bar_rows.append(row)

bar_summary = pd.DataFrame(bar_rows)
bar_summary.to_csv(OUT / "slide04_module_timebin_means_by_sis3.csv", index=False)

# -----------------------------
# 3. Statistics for merged comparisons
# -----------------------------
comparisons = [
    {
        "key": "gt3_vs_0_1",
        "label": ">3y minus 0–1y",
        "late_bins": [">3y"],
        "early_bins": ["0–1y"],
        "effect_direction_label": ">3y − 0–1y",
    },
    {
        "key": "gt3_vs_1_3",
        "label": ">3y minus 1–3y",
        "late_bins": [">3y"],
        "early_bins": ["1–3y"],
        "effect_direction_label": ">3y − 1–3y",
    },
    {
        "key": "1_3_vs_0_1",
        "label": "1–3y minus 0–1y",
        "late_bins": ["1–3y"],
        "early_bins": ["0–1y"],
        "effect_direction_label": "1–3y − 0–1y",
    },
]

forest_rows = []
seed_counter = 0
for sis3_group in ["Lower SIS3 (≤63)", "Higher SIS3 (>63)"]:
    subg = score_df[score_df["sis3_group"] == sis3_group].copy()
    for comp in comparisons:
        for i, pathway in enumerate(PATHWAY_ORDER_FIXED):
            late_vals = pd.to_numeric(subg.loc[subg["time_bin"].isin(comp["late_bins"]), pathway], errors="coerce").dropna()
            early_vals = pd.to_numeric(subg.loc[subg["time_bin"].isin(comp["early_bins"]), pathway], errors="coerce").dropna()
            diff, lo, hi, p = bootstrap_mean_diff_stats(late_vals, early_vals, seed=RANDOM_SEED + seed_counter)
            seed_counter += 1
            forest_rows.append({
                "SIS3 group": sis3_group,
                "Comparison key": comp["key"],
                "Comparison": comp["label"],
                "Effect direction label": comp["effect_direction_label"],
                "Biological module": pathway,
                "Biological module, short": PATHWAY_SHORT.get(pathway, pathway),
                "late mean": float(late_vals.mean()) if len(late_vals) else np.nan,
                "early mean": float(early_vals.mean()) if len(early_vals) else np.nan,
                "effect": diff,
                "ci_low": lo,
                "ci_high": hi,
                "P value": p,
                "P value formatted": fmt_p3(p),
            })

forest_stats = pd.DataFrame(forest_rows)
forest_stats.to_csv(OUT / "slide04_module_time_merged_forest_stats.csv", index=False)

# -----------------------------
# 4. Figure 4A and 4B: Bar figures
# -----------------------------
def make_timebar_figure(sis3_group, color_key, out_stem, panel_title):
    plot_df = bar_summary[bar_summary["SIS3 group"] == sis3_group].copy()
    ylabels = [f"{i+1:02d}. {p}" for i, p in enumerate(PATHWAY_ORDER_FIXED)]
    y = np.arange(len(PATHWAY_ORDER_FIXED))

    fig = plt.figure(figsize=(14.4, 8.9))
    gs = GridSpec(3, 3, figure=fig, height_ratios=[0.42, 4.95, 1.60], width_ratios=[1,1,1], hspace=0.10, wspace=0.18)

    title_ax = fig.add_subplot(gs[0, :])
    title_ax.axis("off")
    title_ax.text(0.00, 0.72, panel_title, ha="left", va="center", fontsize=13.0, fontweight="bold", color=COLORS["black"], transform=title_ax.transAxes)

    axes = []
    all_vals = []
    for t in TIME_ORDER:
        all_vals.extend(plot_df[f"{t}, mean"].tolist())
    xlim = 0.6

    for j, t in enumerate(TIME_ORDER):
        ax = fig.add_subplot(gs[1, j], sharey=axes[0] if axes else None)
        axes.append(ax)
        vals = plot_df[f"{t}, mean"].values.astype(float)
        ax.barh(y, vals, height=0.78, color=COLORS[color_key])
        ax.axvline(0, color=COLORS["black"], lw=0.8)
        ax.set_xlim(-xlim, xlim)
        ax.set_title(f"{t}\n(n={int(plot_df[f'{t}, n'].iloc[0])})", pad=7, fontweight="bold")
        ax.grid(axis="x", color=COLORS["verylight"], lw=0.7)
        ax.set_axisbelow(True)
        ax.tick_params(axis="both", length=3, color=COLORS["gray"])
        if j == 0:
            ax.set_yticks(y)
            ax.set_yticklabels(ylabels)
            ax.set_ylabel("Biological module")
        else:
            ax.set_yticks(y)
            ax.tick_params(axis="y", labelleft=False)
        ax.invert_yaxis()
        ax.set_xlabel("Mean standardized biological-module value")

    cap_ax = fig.add_subplot(gs[2, :])
    cap_ax.axis("off")
    caption = (
        "Bars show mean standardized biological-module values within 0–1 year, 1–3 years, and more than 3 years after stroke. "
        "Positive values indicate higher standardized biological-module values and negative values indicate lower standardized biological-module values relative to the cohort mean."
    )
    cap_ax.text(0.00, 0.60, wrap_caption(caption, width=180), ha="left", va="top", fontsize=8.4, color=COLORS["black"], linespacing=1.18, transform=cap_ax.transAxes)

    fig.subplots_adjust(left=0.33, right=0.985, top=0.93, bottom=0.10)
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(OUT / f"{out_stem}.{ext}", bbox_inches="tight")
    plt.close(fig)

make_timebar_figure("Lower SIS3 (≤63)", "lower", "figure4_low_sis3_timebin_module_values_barplot_journal_style_v8", "Figure 4A. Biological-Module Values Across Time Bins in Lower-SIS3 Patients")
make_timebar_figure("Higher SIS3 (>63)", "higher", "figure4_high_sis3_timebin_module_values_barplot_journal_style_v8", "Figure 4B. Biological-Module Values Across Time Bins in Higher-SIS3 Patients")

# -----------------------------
# 5. Figure 4C and 4D: Forest figures
# -----------------------------
def make_forest_figure(sis3_group, out_stem, figure_title):
    plot_df = forest_stats[forest_stats["SIS3 group"] == sis3_group].copy()
    order_map = {p: i for i, p in enumerate(PATHWAY_ORDER_FIXED)}
    plot_df["order"] = plot_df["Biological module"].map(order_map)

    # fixed comparison order requested by user
    comparison_order = ["gt3_vs_0_1", "gt3_vs_1_3", "1_3_vs_0_1"]
    comparison_titles = {
        "gt3_vs_0_1": ">3y − 0–1y",
        "gt3_vs_1_3": ">3y − 1–3y",
        "1_3_vs_0_1": "1–3y − 0–1y",
    }

    y = np.arange(len(PATHWAY_ORDER_FIXED))
    module_labels = [PATHWAY_SHORT.get(p, p) for p in PATHWAY_ORDER_FIXED]
    number_labels = [f"{i+1:02d}" for i in range(len(PATHWAY_ORDER_FIXED))]

    effect_vals = plot_df["effect"].tolist() + plot_df["ci_low"].tolist() + plot_df["ci_high"].tolist()
    xlim = symmetric_axis_limit(effect_vals, min_limit=0.8, step=0.2, pad=0.06)

    fig = plt.figure(figsize=(24.0, 8.9))
    gs = GridSpec(
        3, 11,
        figure=fig,
        height_ratios=[0.42, 5.0, 1.55],
        width_ratios=[0.55, 4.05, 2.55, 2.15, 1.05, 2.55, 2.15, 1.05, 2.55, 2.15, 1.05],
        hspace=0.10,
        wspace=0.06,
    )

    # Title
    title_ax = fig.add_subplot(gs[0, :])
    title_ax.axis("off")
    title_ax.text(
        0.00, 0.72, figure_title,
        ha="left", va="center", fontsize=13.0, fontweight="bold",
        color=COLORS["black"], transform=title_ax.transAxes
    )

    # Left columns
    ax_no = fig.add_subplot(gs[1, 0])
    ax_mod = fig.add_subplot(gs[1, 1], sharey=ax_no)

    text_axes = [ax_no, ax_mod]
    forest_axes = []
    est_axes = []
    p_axes = []

    # Prepare pairwise comparison axes
    for idx, comp_key in enumerate(comparison_order):
        forest_ax = fig.add_subplot(gs[1, 2 + idx*3], sharey=ax_no)
        est_ax = fig.add_subplot(gs[1, 3 + idx*3], sharey=ax_no)
        p_ax = fig.add_subplot(gs[1, 4 + idx*3], sharey=ax_no)
        forest_axes.append(forest_ax)
        est_axes.append(est_ax)
        p_axes.append(p_ax)
        text_axes.extend([est_ax, p_ax])

    # Common y layout and alternating row shading
    # Use subtly darker alternating-row shading from comparison block 1 to 3
    # so reviewers can more easily track the 3 time-bin contrasts.
    group_shades = {
        "left": "#F3F1ED",   # Figure 2-style alternating row shade
        0: "#F3F1ED",
        1: "#F3F1ED",
        2: "#F3F1ED",
    }

    for ax in [ax_no, ax_mod]:
        ax.set_ylim(len(PATHWAY_ORDER_FIXED)-0.5, -0.7)
        for i in range(len(PATHWAY_ORDER_FIXED)):
            if i % 2 == 1:
                ax.axhspan(i-0.5, i+0.5, color=group_shades["left"], zorder=0)

    for idx in range(3):
        for ax in [forest_axes[idx], est_axes[idx], p_axes[idx]]:
            ax.set_ylim(len(PATHWAY_ORDER_FIXED)-0.5, -0.7)
            for i in range(len(PATHWAY_ORDER_FIXED)):
                if i % 2 == 1:
                    ax.axhspan(i-0.5, i+0.5, color=group_shades[idx], zorder=0)

    # No. column
    ax_no.set_xlim(0, 1)
    ax_no.axis("off")
    ax_no.text(0.00, 1.01, "No.", transform=ax_no.transAxes, ha="left", va="bottom",
               fontsize=10.0, fontweight="bold", color=COLORS["black"])
    for yi, txt in zip(y, number_labels):
        ax_no.text(0.00, yi, txt, ha="left", va="center", fontsize=9.0, color=COLORS["black"])

    # Module column
    ax_mod.set_xlim(0, 1)
    ax_mod.axis("off")
    ax_mod.text(0.00, 1.01, "Biological module", transform=ax_mod.transAxes, ha="left", va="bottom",
                fontsize=10.0, fontweight="bold", color=COLORS["black"])
    for yi, txt in zip(y, module_labels):
        ax_mod.text(0.00, yi, txt, ha="left", va="center", fontsize=9.0, color=COLORS["black"])

    # Comparison blocks
    for idx, comp_key in enumerate(comparison_order):
        sdf = plot_df[plot_df["Comparison key"] == comp_key].copy().sort_values("order")
        forest_ax = forest_axes[idx]
        est_ax = est_axes[idx]
        p_ax = p_axes[idx]

        # forest axis
        forest_ax.hlines(y, sdf["ci_low"].values, sdf["ci_high"].values, color=COLORS["effect"], lw=1.4, zorder=2)
        forest_ax.plot(sdf["effect"].values, y, marker="s", linestyle="None", color=COLORS["effect"], markersize=5.0, zorder=3)
        forest_ax.axvline(0, color=COLORS["gray"], lw=0.8, ls=":")
        forest_ax.set_xlim(-xlim, xlim)
        forest_ax.grid(axis="x", color=COLORS["verylight"], lw=0.7)
        forest_ax.set_axisbelow(True)
        forest_ax.tick_params(axis="x", length=3, color=COLORS["gray"], labelsize=8.8)
        forest_ax.spines["left"].set_visible(True)
        forest_ax.spines["left"].set_linewidth(0.8)
        forest_ax.spines["left"].set_color(COLORS["black"])
        if idx == 0:
            forest_ax.tick_params(axis="y", left=True, labelleft=False, length=3, color=COLORS["gray"])
        else:
            forest_ax.tick_params(axis="y", left=True, labelleft=False, length=3, color=COLORS["gray"])
        forest_ax.set_xlabel("Mean difference (95% CI)")
        forest_ax.text(0.5, 1.06, comparison_titles[comp_key], transform=forest_ax.transAxes,
                       ha="center", va="bottom", fontsize=10.0, fontweight="bold", color=COLORS["black"])
        forest_ax.text(0.00, 1.01, "Mean difference (95% CI)", transform=forest_ax.transAxes,
                       ha="left", va="bottom", fontsize=9.6, fontweight="bold", color=COLORS["black"])

        # estimate text column
        est_ax.set_xlim(0, 1)
        est_ax.axis("off")
        est_ax.text(0.00, 1.01, "Estimate (95% CI)", transform=est_ax.transAxes, ha="left", va="bottom",
                    fontsize=9.6, fontweight="bold", color=COLORS["black"])
        for yi, eff, lo, hi in zip(y, sdf["effect"].values, sdf["ci_low"].values, sdf["ci_high"].values):
            est_ax.text(0.00, yi, f"{eff:.2f} ({lo:.2f} to {hi:.2f})", ha="left", va="center", fontsize=8.9, color=COLORS["black"])

        # p value text column
        p_ax.set_xlim(0, 1)
        p_ax.axis("off")
        p_ax.text(0.00, 1.01, "P value", transform=p_ax.transAxes, ha="left", va="bottom",
                  fontsize=9.6, fontweight="bold", color=COLORS["black"])
        for yi, pv in zip(y, sdf["P value formatted"].values):
            p_ax.text(0.00, yi, pv, ha="left", va="center", fontsize=8.9, color=COLORS["black"])

    # caption
    cap_ax = fig.add_subplot(gs[2, :])
    cap_ax.axis("off")
    caption = (
        "Forest plots show pairwise time-bin comparisons of standardized biological-module values within each SIS3 group. "
        "The 3 contrasts are more than 3 years minus 0–1 year, more than 3 years minus 1–3 years, and 1–3 years minus 0–1 year. "
        "Positive values indicate higher standardized biological-module values in the later time bin. Error bars indicate bootstrap 95% CIs, "
        "and P values were estimated by bootstrap resampling."
    )
    cap_ax.text(
        0.00, 0.60,
        wrap_caption(caption, width=255),
        ha="left", va="top", fontsize=8.4, color=COLORS["black"],
        linespacing=1.18, transform=cap_ax.transAxes
    )

    fig.subplots_adjust(left=0.03, right=0.995, top=0.93, bottom=0.10)
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(OUT / f"{out_stem}.{ext}", bbox_inches="tight")
    plt.close(fig)

make_forest_figure("Lower SIS3 (≤63)", "figure4_low_sis3_timebin_module_values_pairwise_forest_journal_style_v8", "Figure 4C. Time-Bin Effects on Biological-Module Values in Lower-SIS3 Patients")
make_forest_figure("Higher SIS3 (>63)", "figure4_high_sis3_timebin_module_values_pairwise_forest_journal_style_v8", "Figure 4D. Time-Bin Effects on Biological-Module Values in Higher-SIS3 Patients")

print("\nCreated Slide 4 outputs:")
for p in sorted(OUT.glob("figure4_*")):
    print(" -", p.name)


Current folder: /content
CSV files found:
 - C_NPX_data.csv
 - C_patient_data.csv
 - NAME_OID.csv
 - strokecog_literature_aligned_pathway_framework.csv
 - strokecog_literature_aligned_protein_pathway_assignment_summary.csv

Proteins before complete-case filter: 1196
Proteins retained after complete-case filter: 1011

Created Slide 4 outputs:
 - figure4_high_sis3_timebin_barplot_journal_style_v7.pdf
 - figure4_high_sis3_timebin_barplot_journal_style_v7.png
 - figure4_high_sis3_timebin_barplot_journal_style_v7.svg
 - figure4_high_sis3_timebin_pairwise_forest_journal_style_v7.pdf
 - figure4_high_sis3_timebin_pairwise_forest_journal_style_v7.png
 - figure4_high_sis3_timebin_pairwise_forest_journal_style_v7.svg
 - figure4_low_sis3_timebin_barplot_journal_style_v7.pdf
 - figure4_low_sis3_timebin_barplot_journal_style_v7.png
 - figure4_low_sis3_timebin_barplot_journal_style_v7.svg
 - figure4_low_sis3_timebin_pairwise_forest_journal_style_v7.pdf
 - figure4_low_sis3_timebin_pairwise_forest_jour